In [24]:
%pip install nltk

import re
import sys
import bisect
import timeit
import math
import os
# File configuration
GROUP_NAME="Information_Retrieval_Group_Project"
INPUT_FILE="cran.all.1400"
OUTPUT_FILE=GROUP_NAME+"_processed.all"
STOPWORD_FILE="stopwords.txt"


# Splits text into word tokens
def tokenize(text):
    """Return alphabetic tokens from the given text."""

    tokens=re.findall(r"[A-Za-z]+",text)
    return tokens


# Applies Porter stemming
def stem(tokens,stemmer):
    """Stem every token."""

    stemmed_tokens=[]

    for token in tokens:
        stemmed_tokens.append(stemmer.stem(token))

    return stemmed_tokens


# Removes stop words
def remove_stopwords(tokens,stopwords):
    """Remove tokens present in the stop-word set."""

    result=[]

    for token in tokens:
        if token not in stopwords:
            result.append(token)

    return result


# Converts tokens to lowercase
def normalize(tokens):
    """Convert every token to lowercase."""

    return [token.lower() for token in tokens]


# Loads stop words from the file
def load_stopwords(filename):
    """Load one stop word from each line."""

    stopwords=set()

    try:
        with open(filename,"r",encoding="utf-8") as file:
            for line in file:
                word=line.strip().lower()

                if word:
                    stopwords.add(word)

    except FileNotFoundError:
        print("[ERROR] Stop-word file not found:",filename)
        sys.exit(1)

    print("[DEBUG] Loaded",len(stopwords),"stop words")

    return stopwords


# Processes one document
def process_document(text,stemmer,stopwords):
    """Tokenize,normalize,remove stop words and stem the text."""

    # Tokenization
    tokens=tokenize(text)
    print("[DEBUG] Tokens before processing:",len(tokens))

    # Normalization
    tokens=normalize(tokens)

    # Stop-word removal
    tokens=remove_stopwords(tokens,stopwords)
    print("[DEBUG] Tokens after stop-word removal:",len(tokens))

    # Stemming
    tokens=stem(tokens,stemmer)
    print("[DEBUG] Final tokens:",len(tokens))

    return tokens


# Processes the Cranfield collection
def preprocess_collection(input_file,output_file,stemmer,stopwords):
    """Process the title and abstract of every document."""

    current_doc_id=None
    current_section=None
    current_text=[]
    documents_processed=0

    try:
        input_fp=open(input_file,"r",encoding="utf-8")
    except FileNotFoundError:
        print("[ERROR] Input file not found:",input_file)
        sys.exit(1)

    output_fp=open(output_file,"w",encoding="utf-8")

    print("[DEBUG] Starting preprocessing:")
    print("[DEBUG] Input:",input_file)
    print("[DEBUG] Output:",output_file)

    # Writes the current document
    def process_current_document():
        nonlocal current_doc_id
        nonlocal current_text
        nonlocal documents_processed

        if current_doc_id is None:
            return

        # Combines the title and abstract
        text=" ".join(current_text)

        print("\n[DEBUG] Processing document:",current_doc_id)

        tokens=process_document(
            text,
            stemmer,
            stopwords
        )

        # Writes the document ID and tokens
        output_fp.write(".I "+current_doc_id+"\n")
        output_fp.write(".S\n")
        output_fp.write(" ".join(tokens)+"\n")

        documents_processed+=1
        current_text=[]

    # Reads the collection line by line
    for line in input_fp:
        line=line.rstrip("\n")

        # Starts a new document
        if line.startswith(".I"):
            process_current_document()

            parts=line.split()

            if len(parts)>=2:
                current_doc_id=parts[1]
            else:
                print("[WARNING] Invalid .I line:",line)
                current_doc_id=None

            current_section="I"

        # Starts the title
        elif line.startswith(".T"):
            current_section="T"

        # Starts the abstract
        elif line.startswith(".W"):
            current_section="W"

        # Ignores the author section
        elif line.startswith(".A"):
            current_section="A"

        else:
            # Stores only title and abstract text
            if current_section=="T" or current_section=="W":
                current_text.append(line)

    # Writes the last document
    process_current_document()

    input_fp.close()
    output_fp.close()

    print("[DEBUG] Preprocessing complete")
    print("[DEBUG] Documents processed:",documents_processed)
    print("[DEBUG] Output file:",output_file)


# Runs the preprocessing program
def main():
    print(" Cranfield Text Preprocessor")

    try:
        from nltk.stem import PorterStemmer
    except ImportError:
        print("[ERROR] NLTK is not installed.")
        print("Install using:")
        print("    pip install nltk")
        sys.exit(1)

    stemmer=PorterStemmer()
    stopwords=load_stopwords(STOPWORD_FILE)

    preprocess_collection(
        INPUT_FILE,
        OUTPUT_FILE,
        stemmer,
        stopwords
    )


if __name__=="__main__":
    main()

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
 Cranfield Text Preprocessor
[DEBUG] Loaded 358 stop words
[DEBUG] Starting preprocessing:
[DEBUG] Input: cran.all.1400
[DEBUG] Output: Information_Retrieval_Group_Project_processed.all

[DEBUG] Processing document: 1
[DEBUG] Tokens before processing: 150
[DEBUG] Tokens after stop-word removal: 75
[DEBUG] Final tokens: 75

[DEBUG] Processing document: 2
[DEBUG] Tokens before processing: 211
[DEBUG] Tokens after stop-word removal: 122
[DEBUG] Final tokens: 122

[DEBUG] Processing document: 3
[DEBUG] Tokens before processing: 36
[DEBUG] Tokens after stop-word removal: 25
[DEBUG] Final tokens: 25

[DEBUG] Processing document: 4
[DEBUG] Tokens before processing: 92
[DEBUG] Tokens after stop-word removal: 53
[DEBUG] Final

# Indexing

In [25]:
from collections import defaultdict

GROUP_NAME="Information_Retrieval_Group_Project"

inputfile=GROUP_NAME+"_processed.all"
indexfile=GROUP_NAME+"_cran.index"

In [26]:
# Adds one document to the postings
def adddoc(postings,docid,terms):
    for term in terms:
        postings[term].append(docid)


# Builds the inverted index
def makeindex(name):
    postings=defaultdict(list)
    seenids=set()

    docid=None
    terms=set()
    sawtokens=False

    with open(name,"r",encoding="utf-8") as file:
        for lineno,raw in enumerate(file,1):
            line=raw.strip()

            if not line:
                continue

            # Starts a new document
            if line.startswith(".I "):
                if docid is not None:
                    if not sawtokens:
                        raise ValueError(
                            f"missing .S for document {docid}"
                        )

                    adddoc(postings,docid,terms)

                parts=line.split()

                if len(parts)!=2 or not parts[1].isdigit():
                    raise ValueError(
                        f"invalid .I tag at line {lineno}"
                    )

                docid=int(parts[1])

                if docid in seenids:
                    raise ValueError(
                        f"duplicate document id {docid}"
                    )

                seenids.add(docid)
                terms=set()
                sawtokens=False

            # Starts the token section
            elif line==".S":
                if docid is None or sawtokens:
                    raise ValueError(
                        f"invalid .S tag at line {lineno}"
                    )

                sawtokens=True

            # Rejects unexpected tags
            elif line.startswith("."):
                raise ValueError(
                    f"unknown tag at line {lineno}: {line}"
                )

            # Stores the terms of the current document
            else:
                if docid is None or not sawtokens:
                    raise ValueError(
                        f"tokens outside .S at line {lineno}"
                    )

                terms.update(line.split())

    # Adds the last document
    if docid is not None:
        if not sawtokens:
            raise ValueError(
                f"missing .S for document {docid}"
            )

        adddoc(postings,docid,terms)

    if not seenids:
        raise ValueError("no documents found")

    return dict(postings),max(seenids),len(seenids)

In [27]:
# Writes the index to a file
def saveindex(name,postings,maxid):
    with open(name,"w",encoding="utf-8",newline="\n") as file:
        file.write(f"{len(postings)}, {maxid}\n")

        for term in sorted(postings):
            docs=",".join(map(str,postings[term]))
            file.write(f"{term} {docs}\n")


postings,maxid,doccount=makeindex(inputfile)
saveindex(indexfile,postings,maxid)

print("documents indexed:",doccount)
print("vocabulary size:",len(postings))
print("maximum document id:",maxid)
print("posting pairs:",sum(len(docs) for docs in postings.values()))
print("index file:",indexfile)

documents indexed: 1400
vocabulary size: 4188
maximum document id: 1400
posting pairs: 77782
index file: Information_Retrieval_Group_Project_cran.index


In [28]:
# Checks the generated index
def checkindex(name,postings,maxid):
    with open(name,"r",encoding="utf-8") as file:
        head=file.readline().strip()
        expected=f"{len(postings)}, {maxid}"

        if head!=expected:
            raise AssertionError("incorrect index header")

        lastterm=None
        count=0

        for lineno,raw in enumerate(file,2):
            line=raw.rstrip("\n")

            if " " not in line:
                raise AssertionError(
                    f"invalid index line {lineno}"
                )

            term,docs=line.split(" ",1)

            try:
                docids=[int(doc) for doc in docs.split(",")]
            except ValueError:
                raise AssertionError(
                    f"invalid document id at line {lineno}"
                )

            if lastterm is not None and term<=lastterm:
                raise AssertionError(
                    f"terms not sorted at line {lineno}"
                )

            if docids!=sorted(set(docids)):
                raise AssertionError(
                    f"postings not sorted or duplicated at line {lineno}"
                )

            if docids!=postings.get(term):
                raise AssertionError(
                    f"incorrect postings at line {lineno}"
                )

            if any(doc<1 or doc>maxid for doc in docids):
                raise AssertionError(
                    f"document id out of range at line {lineno}"
                )

            lastterm=term
            count+=1

        if count!=len(postings):
            raise AssertionError(
                "vocabulary size does not match index lines"
            )

    print("index correctness check passed")


checkindex(indexfile,postings,maxid)

index correctness check passed


# Boolean Search

In [29]:
def intersect_postings(a_list,b_list):
    # two-pointer merge for finding common docs
    i,j=0,0
    hits=[]
    while i<len(a_list) and j<len(b_list):
        if a_list[i]==b_list[j]:
            hits.append(a_list[i])
            i+=1
            j+=1
        elif a_list[i]<b_list[j]:
            i+=1
        else:
            j+=1
    return hits

def union_postings(a_list,b_list):
    # two-pointer merge for finding all docs.
    i,j=0,0
    hits=[]
    while i<len(a_list) and j<len(b_list):
        if a_list[i]==b_list[j]:
            hits.append(a_list[i])
            i+=1
            j+=1
        elif a_list[i]<b_list[j]:
            hits.append(a_list[i])
            i+=1
        else:
            hits.append(b_list[j])
            j+=1
    while i<len(a_list):
        hits.append(a_list[i])
        i+=1
    while j<len(b_list):
        hits.append(b_list[j])
        j+=1
    return hits

def process_term(term,stemmer,sw_set):
    # normalize, strip stopwords, and stem a query term.
    parts=tokenize(term)
    parts=normalize(parts)
    parts=remove_stopwords(parts,sw_set)
    parts=stem(parts,stemmer)
    return parts

def run_query(q_str,stemmer,sw_set):
    # parse and execute a simple Boolean query.
    parts=q_str.strip().split()
    if not parts:
        return []
    
    # single term query
    if len(parts)==1:
        word=parts[0]
        tokens=process_term(word,stemmer,sw_set)
        if not tokens:
            return []
        return idx_map.get(tokens[0],[])
    
    # two-term AND/OR query
    if len(parts)==3:
        term1,op,term2=parts
        op=op.upper()
        if op not in ("AND","OR"):
            raise ValueError(f"Unsupported operator: '{op}'")
        
        t1=process_term(term1,stemmer,sw_set)
        t2=process_term(term2,stemmer,sw_set)
        
        p1=idx_map.get(t1[0],[]) if t1 else []
        p2=idx_map.get(t2[0],[]) if t2 else []
        
        if op=="AND":
            return intersect_postings(p1,p2)
        else:
            return union_postings(p1,p2)
    
    raise ValueError("Invalid query format")

def execute_query(query,q_id):
    # running  a query and saving results to <q_id>.txt.
    stemmer=PorterStemmer()
    matches=run_query(query,stemmer,stop_words)
    
    path=f"{q_id}.txt"
    with open(path,"w",encoding="utf-8") as out:
        for doc_id in matches:
            out.write(f"{doc_id}\n")
    
    print(f"Query: '{query}'")
    print(f"Match count: {len(matches)}")
    print(f"Matching Document IDs: {', '.join(map(str, matches))}")
    print(f"Output saved to: {path}\n")
    return matches

In [30]:
# testing with few queries
execute_query("aerodynamic AND slipstream", "q1")
execute_query("boundary OR layer", "q2")
execute_query("experimental AND destalling", "q3")

Query: 'aerodynamic AND slipstream'
Match count: 5
Matching Document IDs: 1, 453, 1064, 1089, 1164
Output saved to: q1.txt

Query: 'boundary OR layer'
Match count: 513
Matching Document IDs: 1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 16, 17, 18, 21, 22, 23, 24, 25, 34, 36, 37, 40, 43, 45, 47, 49, 50, 53, 54, 55, 59, 60, 61, 62, 63, 71, 72, 73, 74, 76, 78, 79, 80, 84, 89, 90, 91, 94, 96, 97, 101, 104, 105, 107, 111, 112, 115, 117, 119, 121, 123, 124, 125, 126, 127, 128, 131, 133, 134, 135, 140, 142, 145, 148, 149, 150, 151, 155, 160, 163, 165, 168, 170, 172, 173, 174, 179, 180, 182, 186, 187, 188, 189, 191, 192, 195, 205, 207, 209, 220, 222, 227, 228, 240, 241, 242, 244, 254, 255, 256, 257, 260, 261, 264, 265, 266, 267, 269, 271, 272, 273, 276, 282, 291, 292, 293, 294, 296, 298, 299, 300, 303, 304, 305, 306, 307, 308, 309, 310, 311, 314, 315, 316, 318, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 352, 353, 355

[1, 484]

# immplementing binary search as the index lists are sorted

In [31]:

# two lists are intersected using binary search
def intersect_binary(a_list, b_list):
    if len(a_list)>len(b_list):
        a_list, b_list=b_list, a_list
        
    hits=[]
    left=0
    for doc_id in a_list:
        pos=bisect.bisect_left(b_list, doc_id, lo=left)
        if pos<len(b_list) and b_list[pos]==doc_id:
            hits.append(doc_id)
            left=pos+1
        else:
            left=pos
    return hits

# two lists are unioned using binary search
def union_binary(a_list, b_list):
    if len(a_list)>len(b_list):
        a_list, b_list=b_list, a_list
        
    res=list(b_list)
    left=0
    for doc_id in a_list:
        pos=bisect.bisect_left(res, doc_id, lo=left)
        if pos==len(res) or res[pos]!=doc_id:
            res.insert(pos, doc_id)
            left=pos+1
        else:
            left=pos
    return res

# two terms with different sizes sharing common documents found
sorted_terms=sorted(idx_map.keys(), key=lambda k:len(idx_map[k]))
term_large=sorted_terms[-1]
large_set=set(idx_map[term_large])
term_small=None
for term in sorted_terms:
    if term!=term_large:
        p_curr=idx_map[term]
        if len(large_set.intersection(p_curr))>0 and len(p_curr)<len(idx_map[term_large])//10:
            term_small=term
            break
if term_small is None:
    term_small=sorted_terms[0]

p_small=idx_map[term_small]
p_large=idx_map[term_large]

print(f"postings list size for '{term_small}': {len(p_small)}")
print(f"postings list size for '{term_large}': {len(p_large)}")

twopointer_and=intersect_postings(p_small, p_large)
and_binary=intersect_binary(p_small, p_large)
twopointer_or=union_postings(p_small, p_large)
or_binary=union_binary(p_small, p_large)

print(f"and results are identical: {twopointer_and==and_binary}")
print(f"or results are identical: {twopointer_or==or_binary}")
print(f"intersecting documents: {and_binary}")

# intersection and union are considered over 10000 runs
time_twopointer=timeit.timeit(lambda:intersect_postings(p_small, p_large), number=10000)
time_binary=timeit.timeit(lambda:intersect_binary(p_small, p_large), number=10000)
time_union=timeit.timeit(lambda:union_postings(p_small, p_large), number=10000)
time_orbinary=timeit.timeit(lambda:union_binary(p_small, p_large), number=10000)

print(f"\nperformance comparison (10,000 runs):")
print(f"AND(two-pointer merge): {time_twopointer:.5f} seconds")
print(f"AND(optimized binary):   {time_binary:.5f} seconds")
print(f"AND speedup:             {time_twopointer/time_binary:.2f}x\n")
print(f"OR(two-pointer merge):  {time_union:.5f} seconds")
print(f"OR(binary insertion):   {time_orbinary:.5f} seconds")
print(f"OR speedup:              {time_union/time_orbinary:.2f}x")

postings list size for 'abbrevi': 1
postings list size for 'flow': 730
and results are identical: True
or results are identical: True
intersecting documents: [122]

performance comparison (10,000 runs):
AND(two-pointer merge): 0.06741 seconds
AND(optimized binary):   0.00272 seconds
AND speedup:             24.80x

OR(two-pointer merge):  0.42652 seconds
OR(binary insertion):   0.00773 seconds
OR speedup:              55.17x


# comparing 3 search algorithms : two pointers algorithm,skip lists,binary search

In [32]:

import math
import timeit

# two lists are being intersected using skip pointers
def intersect_skips(a_list, b_list):
    i, j=0, 0
    hits=[]
    step_a=int(math.sqrt(len(a_list)))
    step_b=int(math.sqrt(len(b_list)))
    
    while i<len(a_list) and j<len(b_list):
        if a_list[i]==b_list[j]:
            hits.append(a_list[i])
            i+=1
            j+=1
        elif a_list[i]<b_list[j]:
            if step_a>1 and i+step_a<len(a_list) and a_list[i+step_a]<=b_list[j]:
                while i+step_a<len(a_list) and a_list[i+step_a]<=b_list[j]:
                    i+=step_a
            else:
                i+=1
        else:
            if step_b>1 and j+step_b<len(b_list) and b_list[j+step_b]<=a_list[i]:
                while j+step_b<len(b_list) and b_list[j+step_b]<=a_list[i]:
                    j+=step_b
            else:
                j+=1
    return hits

# two terms with different sizes sharing common documents are being found
all_terms=sorted(idx_map.keys(), key=lambda k:len(idx_map[k]))
big_term=all_terms[-1]
big_docs=set(idx_map[big_term])
small_term=None

for term in all_terms:
    if term!=big_term:
        current=idx_map[term]
        if big_docs.intersection(current):
            if len(current)<len(idx_map[big_term])//10:
                small_term=term
                break

if not small_term:
    small_term=all_terms[0]

small_docs=idx_map[small_term]
large_postings=idx_map[big_term]

print(f"comparing queries for terms: '{small_term}' and '{big_term}'")
print(f"postings list size for '{small_term}': {len(small_docs)}")
print(f"postings list size for '{big_term}': {len(large_postings)}")

twopointer_and=intersect_postings(small_docs, large_postings)
and_skips=intersect_skips(small_docs, large_postings)
print(f"and with skips results are identical: {twopointer_and == and_skips}")

# intersection and union are being benchmarked over 10000 runs
time_twopointer=timeit.timeit(lambda: intersect_postings(small_docs, large_postings), number=10000)
time_binary=timeit.timeit(lambda: intersect_binary(small_docs, large_postings), number=10000)
time_skips=timeit.timeit(lambda: intersect_skips(small_docs, large_postings), number=10000)

time_union=timeit.timeit(lambda: union_postings(small_docs, large_postings), number=10000)
time_orbinary=timeit.timeit(lambda: union_binary(small_docs, large_postings), number=10000)

print(f"\nand performance comparison (10,000 runs):")
print(f"two-pointer merge: {time_twopointer:.5f} seconds")
print(f"optimized binary:   {time_binary:.5f} seconds")
print(f"skip pointer merge: {time_skips:.5f} seconds")

print(f"\nor performance comparison (10,000 runs):")
print(f"two-pointer merge: {time_union:.5f} seconds")
print(f"binary insertion:   {time_orbinary:.5f} seconds")
print(f"skip pointer union: not applicable (skipping cannot be used in union)")


comparing queries for terms: 'abbrevi' and 'flow'
postings list size for 'abbrevi': 1
postings list size for 'flow': 730
and with skips results are identical: True

and performance comparison (10,000 runs):
two-pointer merge: 0.09096 seconds
optimized binary:   0.00305 seconds
skip pointer merge: 0.04451 seconds

or performance comparison (10,000 runs):
two-pointer merge: 0.44426 seconds
binary insertion:   0.00792 seconds
skip pointer union: not applicable (skipping cannot be used in union)


# Testing function for a single query

In [33]:
# checking for single query 
test_query = "aerodynamic AND slipstream"
test_results = run_query(test_query, PorterStemmer(), stop_words)
print(f"\nTesting for a single query")
print(f"Query: '{test_query}'")
print(f"Resulting Doc IDs: {test_results}")


Testing for a single query
Query: 'aerodynamic AND slipstream'
Resulting Doc IDs: [1, 453, 1064, 1089, 1164]


# Fucntion to parse the sample_queries.md and printing the results to results.txt

In [34]:

def parse_sample_queries(filepath):
    queries=[]
    if not os.path.exists(filepath):
        print(f"[ERROR] Testing queries file not found {filepath}")
        return queries

    # this parsing we have used considering the sample_queries.md format
    # if you want to check the one query at a time you can run the above cell by chnaging the query
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line.startswith("|"):
                continue
            parts=[p.strip() for p in line.split("|")]
            if len(parts)<8 or not parts[1].isdigit():
                continue
            
            q_id = int(parts[1])
            q_and = parts[2].replace("`", "")
            q_or = parts[3].replace("`", "")
            exp_and_count = int(parts[4])
            exp_or_count = int(parts[5])
            
            docid_str = parts[6]
            if docid_str:
                exp_and_docs = [int(x.strip()) for x in docid_str.split(",") if x.strip().isdigit()]
            else:
                exp_and_docs = []
                
            queries.append({
                "id": q_id,
                "q_and": q_and,
                "q_or": q_or,
                "exp_and_count": exp_and_count,
                "exp_or_count": exp_or_count,
                "exp_and_docs": exp_and_docs
            })
    return queries

# run the verification for the sample_queries.md
queries = parse_sample_queries("sample_queries.md")
stemmer = PorterStemmer()
output_lines = []
for q in queries:
    q_and = q["q_and"]
    q_or = q["q_or"]
    res_and = run_query(q_and, stemmer, stop_words)
    res_or = run_query(q_or, stemmer, stop_words)
    output_lines.append(f"Query No:{q['id']}\n")
    output_lines.append(f"AND Query:{q_and}")
    output_lines.append(f" Word Match Count: {len(res_and)}")
    output_lines.append(f"Document IDs: {', '.join(map(str, res_and))}")
    output_lines.append(f"")
    output_lines.append(f"OR Query:{q_or}")
    output_lines.append(f"Word Match Count: {len(res_or)}")
    output_lines.append(f"Document IDs: {', '.join(map(str, res_or))}")
    output_lines.append(f"")


for line in output_lines:
    print(line)

# updating the ouput to the result.txt
output_filepath = "result.txt"
with open(output_filepath, "w", encoding="utf-8") as f:
    for line in output_lines:
        f.write(line + "\n")
print(f"\nQuery results written to {output_filepath}")


Query No:1

AND Query:aeroelastic AND aircraft
 Word Match Count: 5
Document IDs: 12, 14, 78, 184, 202

OR Query:aeroelastic OR aircraft
Word Match Count: 84
Document IDs: 12, 14, 29, 47, 51, 75, 76, 78, 100, 141, 172, 184, 195, 202, 209, 220, 237, 245, 251, 253, 284, 311, 328, 345, 364, 374, 390, 415, 416, 453, 486, 497, 658, 685, 721, 724, 725, 726, 729, 746, 747, 781, 791, 792, 804, 810, 811, 836, 875, 878, 882, 883, 884, 908, 909, 911, 914, 917, 925, 948, 1012, 1042, 1051, 1064, 1066, 1089, 1144, 1163, 1165, 1166, 1167, 1168, 1169, 1170, 1197, 1239, 1246, 1300, 1328, 1331, 1332, 1334, 1361, 1380

Query No:2

AND Query:dynamics AND effects
 Word Match Count: 34
Document IDs: 110, 140, 190, 201, 210, 286, 290, 297, 328, 342, 395, 531, 650, 714, 766, 783, 792, 858, 859, 905, 939, 953, 1001, 1004, 1008, 1009, 1066, 1144, 1165, 1203, 1289, 1296, 1321, 1331

OR Query:dynamics OR effects
Word Match Count: 585
Document IDs: 1, 2, 4, 7, 8, 9, 11, 14, 21, 23, 24, 25, 26, 27, 29, 32, 33, 36, 